# Pre-processing Parking Violations

Dieses Notebook lädt die NYC Parking Violations Rohdaten der Fiskaljahre 2023, 2024 und 2025 aus dem HDFS, vereinheitlicht die Datenstruktur, bereinigt zentrale Felder und speichert die verarbeiteten Daten als Parquet-Dateien im HDFS.

## Ziel

- Rohdaten aus HDFS laden
- Fiscal Year ergänzen
- zentrale Spalten auswählen
- Spaltennamen vereinheitlichen
- Issue Date parsen
- zusätzliche Datumsfelder ableiten
- fehlende Werte behandeln
- bereinigte Daten als Parquet speichern

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, to_date, year, month, dayofweek,
    trim, upper, coalesce
)
import re

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Preprocessing") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 18:11:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/09 18:11:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


26/05/09 18:11:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
!hdfs dfs -ls -R /parking_violations/raw

drwxr-xr-x   - cluster supergroup          0 2026-05-07 21:53 /parking_violations/raw/2023
-rw-r--r--   1 cluster supergroup 4025724719 2026-05-07 21:53 /parking_violations/raw/2023/parking_violations_2023.csv
drwxr-xr-x   - cluster supergroup          0 2026-05-07 21:08 /parking_violations/raw/2024
-rw-r--r--   1 cluster supergroup 3002584102 2026-05-07 21:08 /parking_violations/raw/2024/parking_violations_2024.csv
drwxr-xr-x   - cluster supergroup          0 2026-05-07 20:40 /parking_violations/raw/2025
-rw-r--r--   1 cluster supergroup 3064758292 2026-05-07 20:40 /parking_violations/raw/2025/parking_violations_2025.csv


In [3]:
raw_paths = {
    2023: "hdfs:///parking_violations/raw/2023/parking_violations_2023.csv",
    2024: "hdfs:///parking_violations/raw/2024/parking_violations_2024.csv",
    2025: "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
}

processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

raw_paths, processed_path

({2023: 'hdfs:///parking_violations/raw/2023/parking_violations_2023.csv',
  2024: 'hdfs:///parking_violations/raw/2024/parking_violations_2024.csv',
  2025: 'hdfs:///parking_violations/raw/2025/parking_violations_2025.csv'},
 'hdfs:///parking_violations/processed/parking_violations_cleaned')

In [4]:
dfs = []

for fiscal_year, path in raw_paths.items():
    df_year = spark.read.csv(
        path,
        header=True,
        inferSchema=True
    ).withColumn("Fiscal Year", lit(fiscal_year))
    
    dfs.append(df_year)

df_raw = dfs[0]
for df_next in dfs[1:]:
    df_raw = df_raw.unionByName(df_next)

df_raw.groupBy("Fiscal Year").count().orderBy("Fiscal Year").show()

26/05/09 18:12:22 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 18:12:37 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 18:12:52 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 18:13:07 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 18:13:22 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 18:13:37 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure th

KeyboardInterrupt: 

26/05/09 18:16:37 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


In [5]:
!curl -s http://bdlc-012.bdlc.ls.eee.intern:8080 | grep -i -E "workers|alive|cores|memory"

              <li><strong>Alive Workers:</strong> 1</li>
              <li><strong>Cores in use:</strong> 12 Total,
              <li><strong>Memory in use:</strong>
              <li><strong>Status:</strong> ALIVE</li>
            <span class="collapse-aggregated-workers collapse-table" onClick="collapseTable('collapse-aggregated-workers','aggregated-workers')">
                <a>Workers (1)</a>
            <div class="aggregated-workers collapsible-table">
      <thead><th width="" class="">Worker Id</th><th width="" class="">Address</th><th width="" class="">State</th><th width="" class="">Cores</th><th width="" class="">Memory</th><th width="" class="">Resources</th></thead>
      <td>ALIVE</td>
      <thead><th width="" class="">Application ID</th><th width="" class="">Name</th><th width="" class="">Cores</th><th width="" class="">Memory per Executor</th><th width="" class="">Resources Per Executor</th><th width="" class="">Submitted Time</th><th width="" class="">User</th><th wi

26/05/09 18:16:52 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 18:17:07 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 18:17:22 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


In [6]:
spark.stop()

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, to_date, year, month, dayofweek,
    trim, upper, coalesce
)
import re

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Preprocessing") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "8g") \
    .config("spark.cores.max", "4") \
    .getOrCreate()

spark

In [8]:
test_df = spark.read.csv(
    "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
    header=True,
    inferSchema=False
)

test_df.limit(5).show(truncate=False)

26/05/09 18:19:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+--------+------------------+----------+----------+--------------+-----------------+------------+--------------+------------+------------+------------+-----------------------+------------------+------------------+---------------+-----------+--------------+------------+--------------+-------------------+----------------+---------------------------------+------------+--------------------+-------------------+-------------------+-----------+------------+--------------------+----------------------+--------------------+------------------+-------------+---------------------+------------+------------+--------------+-------------------+------------------------------+---------------------------------+-----------------+------------------------+
|Summons Number|Plate ID|Registration State|Plate Type|Issue Date|Violation Code|Vehicle Body Type|Vehicle Make|Issuing Agency|Street Code1|Street Code2|Street Code3|Vehicle Expiration Date|Violation Location|Violation Precinct|Issuer Precin

In [9]:
raw_paths = {
    2023: "hdfs:///parking_violations/raw/2023/parking_violations_2023.csv",
    2024: "hdfs:///parking_violations/raw/2024/parking_violations_2024.csv",
    2025: "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
}

processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

dfs = []

for fiscal_year, path in raw_paths.items():
    df_year = spark.read.csv(
        path,
        header=True,
        inferSchema=False
    ).withColumn("Fiscal Year", lit(fiscal_year))
    
    dfs.append(df_year)

df_raw = dfs[0]
for df_next in dfs[1:]:
    df_raw = df_raw.unionByName(df_next)

df_raw.select("Fiscal Year").groupBy("Fiscal Year").count().orderBy("Fiscal Year").show()

[Stage 5:========================================================>(75 + 1) / 76]

+-----------+--------+
|Fiscal Year|   count|
+-----------+--------+
|       2023|21563238|
|       2024|16101101|
|       2025|16559243|
+-----------+--------+



In [10]:
selected_columns = [
    "Fiscal Year",
    "Summons Number",
    "Plate ID",
    "Registration State",
    "Issue Date",
    "Violation Time",
    "Violation County",
    "Violation Precinct",
    "Street Name",
    "Vehicle Make",
    "Vehicle Body Type",
    "Violation Code",
    "Violation Description"
]

existing_columns = [c for c in selected_columns if c in df_raw.columns]

df_selected = df_raw.select(existing_columns)

df_selected.show(10, truncate=False)

+-----------+--------------+--------+------------------+----------+--------------+----------------+------------------+------------------+------------+-----------------+--------------+---------------------+
|Fiscal Year|Summons Number|Plate ID|Registration State|Issue Date|Violation Time|Violation County|Violation Precinct|Street Name       |Vehicle Make|Vehicle Body Type|Violation Code|Violation Description|
+-----------+--------------+--------+------------------+----------+--------------+----------------+------------------+------------------+------------+-----------------+--------------+---------------------+
|2023       |1484697303    |JER1863 |NY                |06/10/2022|1037A         |NY              |10                |W 28TH ST         |TOYOT       |SDN              |67            |NULL                 |
|2023       |1484697315    |KEV4487 |NY                |06/13/2022|1045A         |NY              |10                |27TH DR           |JEEP        |SUBN             |51      

In [11]:
def normalize_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = name.strip("_")
    return name

df_clean_names = df_selected.toDF(*[normalize_column_name(c) for c in df_selected.columns])

df_clean_names.columns

['fiscal_year',
 'summons_number',
 'plate_id',
 'registration_state',
 'issue_date',
 'violation_time',
 'violation_county',
 'violation_precinct',
 'street_name',
 'vehicle_make',
 'vehicle_body_type',
 'violation_code',
 'violation_description']

In [12]:
df_clean = df_clean_names \
    .withColumn("registration_state", upper(trim(col("registration_state")))) \
    .withColumn("vehicle_make", upper(trim(col("vehicle_make")))) \
    .withColumn("vehicle_body_type", upper(trim(col("vehicle_body_type")))) \
    .withColumn("violation_county", upper(trim(col("violation_county")))) \
    .withColumn("street_name", trim(col("street_name"))) \
    .withColumn("violation_description", coalesce(col("violation_description"), lit("Unknown"))) \
    .withColumn("vehicle_make", coalesce(col("vehicle_make"), lit("Unknown"))) \
    .withColumn("vehicle_body_type", coalesce(col("vehicle_body_type"), lit("Unknown"))) \
    .withColumn("violation_county", coalesce(col("violation_county"), lit("Unknown")))

df_clean.show(10, truncate=False)

+-----------+--------------+--------+------------------+----------+--------------+----------------+------------------+------------------+------------+-----------------+--------------+---------------------+
|fiscal_year|summons_number|plate_id|registration_state|issue_date|violation_time|violation_county|violation_precinct|street_name       |vehicle_make|vehicle_body_type|violation_code|violation_description|
+-----------+--------------+--------+------------------+----------+--------------+----------------+------------------+------------------+------------+-----------------+--------------+---------------------+
|2023       |1484697303    |JER1863 |NY                |06/10/2022|1037A         |NY              |10                |W 28TH ST         |TOYOT       |SDN              |67            |Unknown              |
|2023       |1484697315    |KEV4487 |NY                |06/13/2022|1045A         |NY              |10                |27TH DR           |JEEP        |SUBN             |51      

In [13]:
df_clean = df_clean.withColumn(
    "issue_date_parsed",
    to_date(col("issue_date"), "MM/dd/yyyy")
).withColumn(
    "issue_year",
    year(col("issue_date_parsed"))
).withColumn(
    "issue_month",
    month(col("issue_date_parsed"))
).withColumn(
    "issue_weekday",
    dayofweek(col("issue_date_parsed"))
)

df_clean.select(
    "fiscal_year",
    "issue_date",
    "issue_date_parsed",
    "issue_year",
    "issue_month",
    "issue_weekday"
).show(20, truncate=False)

+-----------+----------+-----------------+----------+-----------+-------------+
|fiscal_year|issue_date|issue_date_parsed|issue_year|issue_month|issue_weekday|
+-----------+----------+-----------------+----------+-----------+-------------+
|2023       |06/10/2022|2022-06-10       |2022      |6          |6            |
|2023       |06/13/2022|2022-06-13       |2022      |6          |2            |
|2023       |06/19/2022|2022-06-19       |2022      |6          |1            |
|2023       |06/19/2022|2022-06-19       |2022      |6          |1            |
|2023       |06/19/2022|2022-06-19       |2022      |6          |1            |
|2023       |06/23/2022|2022-06-23       |2022      |6          |5            |
|2023       |06/23/2022|2022-06-23       |2022      |6          |5            |
|2023       |06/20/2022|2022-06-20       |2022      |6          |2            |
|2023       |06/19/2022|2022-06-19       |2022      |6          |1            |
|2023       |06/25/2022|2022-06-25      

In [14]:
from pyspark.sql.functions import sum as spark_sum

df_clean.select(
    spark_sum(col("issue_date_parsed").isNull().cast("int")).alias("missing_issue_date_parsed")
).show()

[Stage 11:======================================================> (74 + 2) / 76]

+-------------------------+
|missing_issue_date_parsed|
+-------------------------+
|                     2930|
+-------------------------+



In [15]:
df_clean_filtered = df_clean \
    .filter(col("summons_number").isNotNull()) \
    .filter(col("violation_code").isNotNull()) \
    .filter(col("issue_date_parsed").isNotNull())

df_clean_filtered.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

[Stage 14:======================================================> (74 + 2) / 76]

+-----------+--------+
|fiscal_year|   count|
+-----------+--------+
|       2023|21563238|
|       2024|16099641|
|       2025|16557773|
+-----------+--------+



In [16]:
df_clean_filtered.write.mode("overwrite") \
    .partitionBy("fiscal_year") \
    .parquet(processed_path)

In [17]:
!hdfs dfs -ls /parking_violations/processed/parking_violations_cleaned

Found 4 items
-rw-r--r--   2 cluster supergroup          0 2026-05-09 20:39 /parking_violations/processed/parking_violations_cleaned/_SUCCESS
drwxr-xr-x   - cluster supergroup          0 2026-05-09 20:39 /parking_violations/processed/parking_violations_cleaned/fiscal_year=2023
drwxr-xr-x   - cluster supergroup          0 2026-05-09 20:39 /parking_violations/processed/parking_violations_cleaned/fiscal_year=2024
drwxr-xr-x   - cluster supergroup          0 2026-05-09 20:39 /parking_violations/processed/parking_violations_cleaned/fiscal_year=2025


In [18]:
df_processed = spark.read.parquet(processed_path)

df_processed.printSchema()

df_processed.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

root
 |-- summons_number: string (nullable = true)
 |-- plate_id: string (nullable = true)
 |-- registration_state: string (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- violation_time: string (nullable = true)
 |-- violation_county: string (nullable = true)
 |-- violation_precinct: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- vehicle_make: string (nullable = true)
 |-- vehicle_body_type: string (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- issue_date_parsed: date (nullable = true)
 |-- issue_year: integer (nullable = true)
 |-- issue_month: integer (nullable = true)
 |-- issue_weekday: integer (nullable = true)
 |-- fiscal_year: integer (nullable = true)

+-----------+--------+
|fiscal_year|   count|
+-----------+--------+
|       2023|21563238|
|       2024|16099641|
|       2025|16557773|
+-----------+--------+



In [ ]:
## Ergebnis

Das Pre-processing wurde erfolgreich durchgeführt. Die bereinigten Daten wurden als Parquet-Dateien im HDFS gespeichert:

`hdfs:///parking_violations/processed/parking_violations_cleaned`

Die Daten sind nach `fiscal_year` partitioniert. Dadurch können spätere Analysen pro Fiskaljahr effizienter ausgeführt werden.

Nach dem Cleaning enthält der Datensatz:

- FY2023: 21'563'238 Zeilen
- FY2024: 16'099'641 Zeilen
- FY2025: 16'557'773 Zeilen

Nur Zeilen mit fehlender `summons_number`, fehlendem `violation_code` oder nicht parsebarem `issue_date` wurden entfernt.